# E-Commerce 360° Analytics — Feature Engineering

## Objective

Feature engineering transforms cleaned data into meaningful business features that support deeper analysis, SQL, Power BI, and future machine-learning tasks.

## Basis for Feature Engineering

Features will be created based on:

1. **EDA Findings** — patterns and issues identified during exploratory analysis.
2. **Business Questions** — metrics required to answer important business problems.
3. **Available Data** — features must be derived from the columns available in the Olist datasets.
4. **Downstream Analysis** — features should be useful for SQL analysis, dashboards, KPIs, and future modeling.

## Techniques We Will Use

- **Feature Creation** — create new variables from existing columns.
- **Date/Time Extraction** — create year, month, quarter, and other time features.
- **Aggregation** — summarize orders, revenue, units, and activity at customer, product, and seller levels.
- **Derived Metrics** — calculate metrics such as delivery days, delivery delay, order value, and revenue.
- **Ratio Features** — create metrics such as average units per order and average order value.
- **Categorical Encoding** — use only when numerical representation is required for future modeling.
- **Scaling/Normalization** — use only when required by a machine-learning algorithm.
- **Outlier Handling** — apply only when extreme values affect the intended analysis.
- **PCA / Dimensionality Reduction** — not required for the current business analysis; will be considered only if a future machine-learning task requires dimensionality reduction.

## What We Will Cover

1. Load Cleaned Data
2. Order Date Features
3. Delivery Features
4. Order Value Features
5. Customer Features
6. Product Features
7. Seller Features
8. Save Feature-Engineered Data

## Key Business Questions

- When are customers placing orders?
- How long does delivery take and which orders are late?
- What is the value of each order?
- Which customers are repeat and high-value customers?
- Which products generate the most sales and revenue?
- Which sellers perform best?
- Which features are required for SQL, Power BI, and future customer/sales analysis?

## Expected Outcome

Create a reusable, business-ready feature dataset based on the findings from EDA rather than creating features randomly.

In [1]:
# Step 1.1 — Import Libraries

import pandas as pd
from pathlib import Path

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# Step 1.2 — Load Processed Datasets

processed_path = Path("../data/processed")

dataframes = {}

for file in processed_path.glob("*.csv"):
    dataframes[file.stem] = pd.read_csv(file)

print(f"Datasets loaded: {len(dataframes)}")

Datasets loaded: 10


In [3]:
# Step 1.3 — Create DataFrame References

customers = dataframes["customers"]
orders = dataframes["orders"]
order_items = dataframes["order_items"]
order_payments = dataframes["order_payments"]
order_reviews = dataframes["order_reviews"]
products = dataframes["products"]
sellers = dataframes["sellers"]

print("Required datasets are ready.")

Required datasets are ready.


## 2. Order Date Features

### Features
- `order_year`
- `order_month`
- `order_quarter`
- `order_day_of_week`

### Why?
To analyze order trends, seasonality, and time-based customer demand.

### Technique
**Date/Time Feature Extraction** — deriving useful time features from `order_purchase_timestamp`.

In [4]:
# Step 2.1 — Convert Order Timestamp

orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"]
)

orders["order_purchase_timestamp"].dtype

dtype('<M8[us]')

In [5]:
# Step 2.2 — Create Order Year

orders["order_year"] = orders["order_purchase_timestamp"].dt.year

orders[["order_purchase_timestamp", "order_year"]].head()

,order_purchase_timestamp,order_year
0,2017-10-02 10:56:33,2017
1,2018-07-24 20:41:37,2018
2,2018-08-08 08:38:49,2018
3,2017-11-18 19:28:06,2017
4,2018-02-13 21:18:39,2018


In [6]:
# Step 2.3 — Create Order Month

orders["order_month"] = orders["order_purchase_timestamp"].dt.month

orders[["order_purchase_timestamp", "order_month"]].head()

,order_purchase_timestamp,order_month
0,2017-10-02 10:56:33,10
1,2018-07-24 20:41:37,7
2,2018-08-08 08:38:49,8
3,2017-11-18 19:28:06,11
4,2018-02-13 21:18:39,2


In [7]:
# Step 2.4 — Create Order Quarter

orders["order_quarter"] = orders["order_purchase_timestamp"].dt.quarter

orders[["order_purchase_timestamp", "order_quarter"]].head()

,order_purchase_timestamp,order_quarter
0,2017-10-02 10:56:33,4
1,2018-07-24 20:41:37,3
2,2018-08-08 08:38:49,3
3,2017-11-18 19:28:06,4
4,2018-02-13 21:18:39,1


In [8]:
# Step 2.5 — Create Order Day of Week

orders["order_day_of_week"] = (
    orders["order_purchase_timestamp"].dt.day_name()
)

orders[[
    "order_purchase_timestamp",
    "order_year",
    "order_month",
    "order_quarter",
    "order_day_of_week"
]].head()

,order_purchase_timestamp,order_year,order_month,order_quarter,order_day_of_week
0,2017-10-02 10:56:33,2017,10,4,Monday
1,2018-07-24 20:41:37,2018,7,3,Tuesday
2,2018-08-08 08:38:49,2018,8,3,Wednesday
3,2017-11-18 19:28:06,2017,11,4,Saturday
4,2018-02-13 21:18:39,2018,2,1,Tuesday


## 3. Delivery Features

### Features
- `delivery_days`
- `carrier_handling_days`
- `delivery_delay_days`
- `is_late`

### Why?
To measure delivery speed, identify delayed orders, and evaluate delivery performance.

### Technique
**Date Difference & Derived Feature Creation** — calculating durations and creating a late-delivery indicator from order and delivery timestamps.

In [9]:
# Step 3.1 — Convert Delivery Timestamps

date_columns = [
    "order_purchase_timestamp",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in date_columns:
    orders[column] = pd.to_datetime(orders[column])

orders[date_columns].dtypes

order_purchase_timestamp         datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [10]:
# Step 3.2 — Create Delivery Days

orders["delivery_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 60 * 60)

orders["delivery_days"] = orders["delivery_days"].round(2)

orders[["order_id", "delivery_days"]].head()

,order_id,delivery_days
0,e481f51cbdc54678b7cc49136f2d6af7,8.44
1,53cdb2fc8bc7dce0b6741e2150273451,13.78
2,47770eb9100c2d0c44946d9cf07ec65d,9.39
3,949d5b44dbf5de918fe9c16f97b45f8a,13.21
4,ad21c59c0840e6cb83a9ceb5573f8159,2.87


In [11]:
# Step 3.3 — Create Carrier Handling Days

orders["carrier_handling_days"] = (
    orders["order_delivered_carrier_date"]
    - orders["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 60 * 60)

orders["carrier_handling_days"] = orders["carrier_handling_days"].round(2)

orders[["order_id", "carrier_handling_days"]].head()

,order_id,carrier_handling_days
0,e481f51cbdc54678b7cc49136f2d6af7,2.37
1,53cdb2fc8bc7dce0b6741e2150273451,1.74
2,47770eb9100c2d0c44946d9cf07ec65d,0.22
3,949d5b44dbf5de918fe9c16f97b45f8a,3.76
4,ad21c59c0840e6cb83a9ceb5573f8159,0.94


In [12]:
# Step 3.4 — Create Delivery Delay Days

orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.total_seconds() / (24 * 60 * 60)

orders["delivery_delay_days"] = orders["delivery_delay_days"].round(2)

orders[["order_id", "delivery_delay_days"]].head()

,order_id,delivery_delay_days
0,e481f51cbdc54678b7cc49136f2d6af7,-7.11
1,53cdb2fc8bc7dce0b6741e2150273451,-5.36
2,47770eb9100c2d0c44946d9cf07ec65d,-17.25
3,949d5b44dbf5de918fe9c16f97b45f8a,-12.98
4,ad21c59c0840e6cb83a9ceb5573f8159,-9.24


In [13]:
# Step 3.5 — Create Late Delivery Indicator

orders["is_late"] = (
    orders["delivery_delay_days"] > 0
).astype(int)

orders[[
    "order_id",
    "delivery_delay_days",
    "is_late"
]].head()

,order_id,delivery_delay_days,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,-7.11,0
1,53cdb2fc8bc7dce0b6741e2150273451,-5.36,0
2,47770eb9100c2d0c44946d9cf07ec65d,-17.25,0
3,949d5b44dbf5de918fe9c16f97b45f8a,-12.98,0
4,ad21c59c0840e6cb83a9ceb5573f8159,-9.24,0


## 4. Order Value Features

### Features
- `item_count`
- `product_value`
- `freight_value`
- `order_value`

### Why?
To measure the monetary value and item volume of each order.

### Technique
**Aggregation & Derived Feature Creation** — summarizing order-item data at the `order_id` level and combining product price with freight value.

In [14]:
# Step 4.1 — Create Order-Level Item Count

order_value_features = (
    order_items
    .groupby("order_id")
    .agg(
        item_count=("order_item_id", "count"),
        product_value=("price", "sum"),
        freight_value=("freight_value", "sum")
    )
    .reset_index()
)

order_value_features.head()

,order_id,item_count,product_value,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14


In [15]:
# Step 4.2 — Create Order Value

order_value_features["order_value"] = (
    order_value_features["product_value"]
    + order_value_features["freight_value"]
)

order_value_features.head()

,order_id,item_count,product_value,freight_value,order_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29,72.19
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93,259.83
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87,216.87
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14,218.04


In [16]:
# Step 4.3 — Merge Order Value Features

orders = orders.merge(
    order_value_features,
    on="order_id",
    how="left"
)

orders[
    [
        "order_id",
        "item_count",
        "product_value",
        "freight_value",
        "order_value"
    ]
].head()

,order_id,item_count,product_value,freight_value,order_value
0,e481f51cbdc54678b7cc49136f2d6af7,1.0,29.99,8.72,38.71
1,53cdb2fc8bc7dce0b6741e2150273451,1.0,118.70,22.76,141.46
2,47770eb9100c2d0c44946d9cf07ec65d,1.0,159.90,19.22,179.12
3,949d5b44dbf5de918fe9c16f97b45f8a,1.0,45.00,27.20,72.20
4,ad21c59c0840e6cb83a9ceb5573f8159,1.0,19.90,8.72,28.62


## 5. Customer Features

### Features
- `customer_order_count`
- `customer_total_spend`
- `customer_average_order_value`
- `customer_first_order_date`
- `customer_last_order_date`

### Why?
To understand customer purchase frequency, spending behavior, and customer activity over time.

### Technique
**Aggregation & Derived Feature Creation** — summarizing order-level data at the `customer_id` level and deriving customer behavior metrics.

In [17]:
# Step 5.1 — Create Customer Order Features

customer_features = (
    orders
    .groupby("customer_id")
    .agg(
        customer_order_count=("order_id", "nunique"),
        customer_total_spend=("order_value", "sum"),
        customer_first_order_date=("order_purchase_timestamp", "min"),
        customer_last_order_date=("order_purchase_timestamp", "max")
    )
    .reset_index()
)

customer_features.head()

,customer_id,customer_order_count,customer_total_spend,customer_first_order_date,customer_last_order_date
0,00012a2ce6f8dcda20d059ce98491703,1,114.74,2017-11-14 16:08:26,2017-11-14 16:08:26
1,000161a058600d5901f007fab4c27140,1,67.41,2017-07-16 09:40:32,2017-07-16 09:40:32
2,0001fd6190edaaf884bcaf3d49edf079,1,195.42,2017-02-28 11:06:43,2017-02-28 11:06:43
3,0002414f95344307404f0ace7a26f1d5,1,179.35,2017-08-16 13:09:20,2017-08-16 13:09:20
4,000379cdec625522490c315e70c7a9fb,1,107.01,2018-04-02 13:42:17,2018-04-02 13:42:17


In [18]:
# Step 5.2 — Create Customer Average Order Value

customer_features["customer_average_order_value"] = (
    customer_features["customer_total_spend"]
    / customer_features["customer_order_count"]
).round(2)

customer_features.head()

,customer_id,customer_order_count,customer_total_spend,customer_first_order_date,customer_last_order_date,customer_average_order_value
0,00012a2ce6f8dcda20d059ce98491703,1,114.74,2017-11-14 16:08:26,2017-11-14 16:08:26,114.74
1,000161a058600d5901f007fab4c27140,1,67.41,2017-07-16 09:40:32,2017-07-16 09:40:32,67.41
2,0001fd6190edaaf884bcaf3d49edf079,1,195.42,2017-02-28 11:06:43,2017-02-28 11:06:43,195.42
3,0002414f95344307404f0ace7a26f1d5,1,179.35,2017-08-16 13:09:20,2017-08-16 13:09:20,179.35
4,000379cdec625522490c315e70c7a9fb,1,107.01,2018-04-02 13:42:17,2018-04-02 13:42:17,107.01


In [19]:
# Step 5.3 — Merge Customer Features

customers = customers.merge(
    customer_features,
    on="customer_id",
    how="left"
)

customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,customer_order_count,customer_total_spend,customer_first_order_date,customer_last_order_date,customer_average_order_value
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,1,146.87,2017-05-16 15:05:35,2017-05-16 15:05:35,146.87
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,1,335.48,2018-01-12 20:48:24,2018-01-12 20:48:24,335.48
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,1,157.73,2018-05-19 16:07:45,2018-05-19 16:07:45,157.73
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,1,173.30,2018-03-13 16:06:38,2018-03-13 16:06:38,173.30
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,1,252.25,2018-07-29 09:51:30,2018-07-29 09:51:30,252.25


## 6. Product Features

### Features
- `product_units_sold`
- `product_revenue`
- `product_average_price`

### Why?
To measure product sales volume, revenue contribution, and average selling price.

### Technique
**Aggregation & Derived Feature Creation** — summarizing order-item data at the `product_id` level.

In [20]:
# Step 6.1 — Create Product Sales Features

product_features = (
    order_items
    .groupby("product_id")
    .agg(
        product_units_sold=("order_id", "count"),
        product_revenue=("price", "sum"),
        product_average_price=("price", "mean")
    )
    .reset_index()
)

product_features.head()

,product_id,product_units_sold,product_revenue,product_average_price
0,00066f42aeeb9f3007548bb9d3f33c38,1,101.65,101.65
1,00088930e925c41fd95ebfe695fd2655,1,129.90,129.90
2,0009406fd7479715e4bef61dd91f2462,1,229.00,229.00
3,000b8f95fcb9e0096488278317764d19,2,117.80,58.90
4,000d9be29b5207b54e86aa1b1ac54872,1,199.00,199.00


In [21]:
# Step 6.2 — Merge Product Features

products = products.merge(
    product_features,
    on="product_id",
    how="left"
)

products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_units_sold,product_revenue,product_average_price
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,1,10.91,10.91
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,1,248.00,248.00
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,1,79.80,79.80
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0,1,112.30,112.30
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0,1,37.90,37.90


## 7. Seller Features

### Features
- `seller_order_count`
- `seller_units_sold`
- `seller_revenue`
- `seller_average_order_value`
- `seller_average_units_per_order`

### Why?
To measure seller sales volume, revenue contribution, and order-level performance.

### Technique
**Aggregation & Derived Feature Creation** — summarizing order-item data at the `seller_id` level and creating performance metrics.

In [22]:
# Step 7.1 — Create Seller Sales Features

seller_features = (
    order_items
    .groupby("seller_id")
    .agg(
        seller_order_count=("order_id", "nunique"),
        seller_units_sold=("order_id", "count"),
        seller_revenue=("price", "sum")
    )
    .reset_index()
)

seller_features.head()

,seller_id,seller_order_count,seller_units_sold,seller_revenue
0,0015a82c2db000af6aaaf3ae2ecb0532,3,3,2685.00
1,001cca7ae9ae17fb1caed9dfb1094831,200,239,25080.03
2,001e6ad469a905060d959994f1b41e4f,1,1,250.00
3,002100f778ceb8431b7a1020ff7ab48f,51,55,1234.50
4,003554e2dce176b5555353e4f3555ac8,1,1,120.00


In [23]:
# Step 7.2 — Create Seller Average Order Value

seller_features["seller_average_order_value"] = (
    seller_features["seller_revenue"]
    / seller_features["seller_order_count"]
).round(2)

seller_features.head()

,seller_id,seller_order_count,seller_units_sold,seller_revenue,seller_average_order_value
0,0015a82c2db000af6aaaf3ae2ecb0532,3,3,2685.00,895.00
1,001cca7ae9ae17fb1caed9dfb1094831,200,239,25080.03,125.40
2,001e6ad469a905060d959994f1b41e4f,1,1,250.00,250.00
3,002100f778ceb8431b7a1020ff7ab48f,51,55,1234.50,24.21
4,003554e2dce176b5555353e4f3555ac8,1,1,120.00,120.00


In [24]:
# Step 7.3 — Create Seller Average Units per Order

seller_features["seller_average_units_per_order"] = (
    seller_features["seller_units_sold"]
    / seller_features["seller_order_count"]
).round(2)

seller_features.head()

,seller_id,seller_order_count,seller_units_sold,seller_revenue,seller_average_order_value,seller_average_units_per_order
0,0015a82c2db000af6aaaf3ae2ecb0532,3,3,2685.00,895.00,1.00
1,001cca7ae9ae17fb1caed9dfb1094831,200,239,25080.03,125.40,1.20
2,001e6ad469a905060d959994f1b41e4f,1,1,250.00,250.00,1.00
3,002100f778ceb8431b7a1020ff7ab48f,51,55,1234.50,24.21,1.08
4,003554e2dce176b5555353e4f3555ac8,1,1,120.00,120.00,1.00


In [25]:
# Step 7.4 — Merge Seller Features

sellers = sellers.merge(
    seller_features,
    on="seller_id",
    how="left"
)

sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state,seller_order_count,seller_units_sold,seller_revenue,seller_average_order_value,seller_average_units_per_order
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP,3,3,218.70,72.90,1.00
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP,40,41,11703.07,292.58,1.02
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ,1,1,158.00,158.00,1.00
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP,1,1,79.99,79.99,1.00
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP,1,1,167.99,167.99,1.00


## 8. Save Feature-Engineered Data

In [26]:
# Step 8.1 — Save Feature-Engineered Datasets

orders.to_csv(processed_path / "orders_feature_engineered.csv", index=False)
customers.to_csv(processed_path / "customers_feature_engineered.csv", index=False)
products.to_csv(processed_path / "products_feature_engineered.csv", index=False)
sellers.to_csv(processed_path / "sellers_feature_engineered.csv", index=False)

print("Feature-engineered datasets saved successfully.")

Feature-engineered datasets saved successfully.


In [27]:
# Step 8.2 — Verify Saved Files

feature_engineered_files = [
    "orders_feature_engineered.csv",
    "customers_feature_engineered.csv",
    "products_feature_engineered.csv",
    "sellers_feature_engineered.csv"
]

for file in feature_engineered_files:
    path = processed_path / file
    print(f"{file}: {path.exists()}")

orders_feature_engineered.csv: True
customers_feature_engineered.csv: True
products_feature_engineered.csv: True
sellers_feature_engineered.csv: True
